In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

from torch.utils.data import Dataset,DataLoader

from tokenizers import ByteLevelBPETokenizer
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace

from torchinfo import summary

САМ МИПЛ (ЧАСТЬ ОБРАБОТКИ СООБЩЕНИЙ ДРУГИХ МИПЛОВ ЕЩЕ НЕ ДОБАВЛЕНА)

In [8]:
class Miple(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, market_features, emotions_count, dropout):
        super(Miple, self).__init__()

        self.emb = nn.Embedding(vocab_size, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.text_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.market_lstm = nn.LSTM(market_features, hidden_size, batch_first=True)
        combined_input_size = (hidden_size * 3) + emotions_count

        #обработка рынка
        self.decision = nn.Sequential(
            nn.Linear(combined_input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.Linear(hidden_size, market_features),
        )

        #обработка эмоций
        self.emotions_choicer = nn.Linear(combined_input_size, emotions_count)

        # общение
        self.combined_to_hidden = nn.Linear(combined_input_size, hidden_size)
        self.text_decoder_lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.chat_message_writer = nn.Linear(hidden_size, vocab_size)

    def forward(self, market_seq, prompt_seq, news_seq, current_emotions,target_seq=None):
            # ембеддинги
            p_emb = self.emb(prompt_seq)
            n_emb = self.emb(news_seq)

            # взятие последнего состояния последнего слоя нейросети
            _, (h_p, _) = self.text_lstm(p_emb)
            _, (h_n, _) = self.text_lstm(n_emb)
            _, (h_m, _) = self.market_lstm(market_seq)

            combined = torch.cat((h_p[-1], h_n[-1], h_m[-1], current_emotions), dim=1)

            trade_logits = self.decision(combined)
            new_emotions = self.emotions_choicer(combined)

            decoder_hidden_init = torch.tanh(self.combined_to_hidden(combined)).unsqueeze(0) # [1, batch_size, hidden_size]
            decoder_cell_init = torch.zeros_hidden_like = torch.zeros_like(decoder_hidden_init)
            dec_states = (decoder_hidden_init, decoder_cell_init)

            if target_seq is not None:
                dec_emb = self.dropout(self.emb(target_seq))
                dec_outputs, _ = self.text_decoder_lstm(dec_emb, dec_states)
                text_logits = self.chat_message_writer(dec_outputs) # [batch_size, seq_len, vocab_size]
            else:
                text_logits = None

            return trade_logits, new_emotions, text_logits


Рассказчик (главный агент, который контролирует правила симуляции)

In [9]:
class StoryTeller:
    def __init__(self, actions: list, randomization_level: int, behavior: str):
        self.actions = actions
        self.randomization_level = randomization_level
        self.behavior = behavior


In [10]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"],
    min_frequency=2
)

files = ["VocabText.txt"]
tokenizer.train(files, trainer)
tokenizer.save("tokenizer.json")


In [11]:
# тестовый прогон
vocab_size = tokenizer.get_vocab_size()
embed_size = 128
hidden_size = 128
market_features = 10
emotions_count = 4
dropout = 0.1

model = Miple(vocab_size, embed_size, hidden_size, market_features, emotions_count, dropout)

batch_size = 8
seq_len_text = 128    # длина текста
seq_len_market = 16  # длина истории рынка


market_input = torch.randn(batch_size, seq_len_market, market_features)
prompt_input = torch.randint(0, vocab_size, (batch_size, seq_len_text))
news_input = torch.randint(0, vocab_size, (batch_size, seq_len_text))
emotions_input = torch.randn(batch_size, emotions_count)


summary(model, input_data=[market_input, prompt_input, news_input, emotions_input])

Layer (type:depth-idx)                   Output Shape              Param #
Miple                                    [8, 10]                   507,486
├─Embedding: 1-1                         [8, 128, 128]             372,480
├─Embedding: 1-2                         [8, 128, 128]             (recursive)
├─LSTM: 1-3                              [8, 128, 128]             132,096
├─LSTM: 1-4                              [8, 128, 128]             (recursive)
├─LSTM: 1-5                              [8, 16, 128]              71,680
├─Sequential: 1-6                        [8, 10]                   --
│    └─Linear: 2-1                       [8, 128]                  49,792
│    └─ReLU: 2-2                         [8, 128]                  --
│    └─Linear: 2-3                       [8, 128]                  16,512
│    └─Linear: 2-4                       [8, 10]                   1,290
├─Linear: 1-7                            [8, 4]                    1,556
├─Linear: 1-8                     